# PPG vs NVDA Statement Retrieval Debug

This notebook diagnoses why statement extraction currently fails for `PPG` while `NVDA` works as a benchmark.

It does **not** modify production code.

In [1]:
import os
import sys
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'Test_Sec_Data' else os.getcwd()
if ROOT not in sys.path:
    sys.path.append(ROOT)

from _01_Data_Analysis.Statement_Data import EdgarFinancials, statement_keywords

EMAIL = 'feb2126@columbia.edu'
STATEMENT_TYPES = ['Income Statement', 'Balance Sheet', 'Cash Flow Statement']
TICKERS = ['NVDA', 'PPG']

print('Loaded modules and configuration.')

Loaded modules and configuration.


In [6]:
import requests



def inspect_filing_mapping(ticker: str, filing_type: str = '10-K', count: int = 1):

    client = EdgarFinancials(email=EMAIL, ticker=ticker)

    filings = client.get_multiple_filings(filing_type=filing_type, count=count)

    filing = filings[0]



    reports_df = filing.reports.to_pandas()

    statements_df = reports_df[reports_df['MenuCategory'] == 'Statements'].copy()

    found_indices = client.find_statement_indices_by_keywords(filing, STATEMENT_TYPES)



    cik = client.company.cik

    accession_no_hyphens = filing.accession_number.replace('-', '')

    excel_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_hyphens}/Financial_Report.xlsx"



    status_code = None

    try:

        r = requests.get(excel_url, headers=client.header, timeout=30)

        status_code = r.status_code

    except Exception:

        status_code = None



    excel_sheets = client.get_excel_from_filing(filing)

    excel_available = len(excel_sheets) > 0



    rows = []

    for statement_type in STATEMENT_TYPES:

        idx = found_indices.get(statement_type)

        shortname = None

        if idx is not None and idx in reports_df.index:

            shortname = reports_df.loc[idx, 'ShortName']



        current_sheet_name = None

        if idx is not None and idx < len(excel_sheets):

            current_sheet_name = excel_sheets[idx][0]



        name_contains_match = None

        if current_sheet_name and isinstance(shortname, str):

            lhs = current_sheet_name.lower().replace('_', ' ')

            rhs = shortname.lower()

            name_contains_match = (lhs in rhs) or (rhs in lhs) or lhs[:24] in rhs



        rows.append({

            'ticker': ticker,

            'accession': filing.accession_number,

            'statement_type': statement_type,

            'report_idx_from_filingsummary': idx,

            'matched_shortname': shortname,

            'excel_url_status': status_code,

            'excel_sheet_count': len(excel_sheets),

            'sheet_name_with_current_logic': current_sheet_name,

            'sheet_name_matches_shortname': name_contains_match,

            'excel_file_available': excel_available

        })



    extracted = client.get_statements_by_type([filing], STATEMENT_TYPES)

    diag_df = pd.DataFrame(rows)

    diag_df['statements_returned_by_current_method'] = len(extracted)



    return diag_df, reports_df, statements_df, excel_sheets, excel_url



print('Inspection helper ready.')

Inspection helper ready.


In [7]:
results = {}

for ticker in TICKERS:

    df_diag, reports_df, statements_df, sheets, excel_url = inspect_filing_mapping(ticker)

    results[ticker] = {

        'diag': df_diag,

        'reports_df': reports_df,

        'statements_df': statements_df,

        'sheets': sheets,

        'excel_url': excel_url

    }



display(pd.concat([results['NVDA']['diag'], results['PPG']['diag']], ignore_index=True))

Could not retrieve Excel file. It may not exist for this filing (0000079879-26-000046). HTTP Status: 404
Could not retrieve Excel file. It may not exist for this filing (0000079879-26-000046). HTTP Status: 404


,ticker,accession,statement_type,report_idx_from_filingsummary,matched_shortname,excel_url_status,excel_sheet_count,sheet_name_with_current_logic,sheet_name_matches_shortname,excel_file_available,statements_returned_by_current_method
0,NVDA,0001045810-25-000023,Income Statement,2,Consolidated Statements of Income,200,91,Consolidated Statements of Inco,True,True,3
1,NVDA,0001045810-25-000023,Balance Sheet,4,Consolidated Balance Sheets,200,91,Consolidated Balance Sheets,True,True,3
2,NVDA,0001045810-25-000023,Cash Flow Statement,8,Consolidated Statements of Cash Flows,200,91,Consolidated Statements of Cash,True,True,3
3,PPG,0000079879-26-000046,Income Statement,2,Consolidated Statement of Income,404,0,None,None,False,0
4,PPG,0000079879-26-000046,Balance Sheet,4,Consolidated Balance Sheet,404,0,None,None,False,0
5,PPG,0000079879-26-000046,Cash Flow Statement,6,Consolidated Statement of Cash Flows,404,0,None,None,False,0


In [4]:
for ticker in TICKERS:
    print('=' * 90)
    print(f'{ticker} | accession: {results[ticker]["diag"].iloc[0]["accession"]}')
    print('- Statement menu index labels and short names:')
    print(results[ticker]['statements_df'][['ShortName']].head(15))
    print('- First 15 Excel sheet names (by positional index):')
    for i, (sheet_name, _) in enumerate(results[ticker]['sheets'][:15]):
        print(f'  {i:2d}: {sheet_name}')

NVDA | accession: 0001045810-25-000023
- Statement menu index labels and short names:
                                           ShortName
2                  Consolidated Statements of Income
3    Consolidated Statements of Comprehensive Income
4                        Consolidated Balance Sheets
5        Consolidated Balance Sheets (Parenthetical)
6    Consolidated Statements of Shareholders' Equity
7  Consolidated Statements of Shareholders' Equit...
8              Consolidated Statements of Cash Flows
- First 15 Excel sheet names (by positional index):
   0: Cover Page
   1: Audit Information
   2: Consolidated Statements of Inco
   3: Consolidated Statements of Comp
   4: Consolidated Balance Sheets
   5: Consolidated Balance Sheets (Pa
   6: Consolidated Statements of Shar
   7: Consolidated Statements of Sh_2
   8: Consolidated Statements of Cash
   9: Organization and Summary of Sig
  10: Business Combination
  11: Stock-Based Compensation
  12: Net Income Per Share
  13: Goodwi

In [8]:
summary_rows = []

for ticker in TICKERS:

    diag = results[ticker]['diag']

    summary_rows.append({

        'ticker': ticker,

        'accession': diag['accession'].iloc[0],

        'excel_url_status': diag['excel_url_status'].iloc[0],

        'excel_file_available': bool(diag['excel_file_available'].iloc[0]),

        'excel_sheet_count': int(diag['excel_sheet_count'].iloc[0]),

        'statements_returned_by_current_method': int(diag['statements_returned_by_current_method'].iloc[0])

    })



summary_df = pd.DataFrame(summary_rows)

display(summary_df)



ppg_row = summary_df[summary_df['ticker'] == 'PPG'].iloc[0]

nvda_row = summary_df[summary_df['ticker'] == 'NVDA'].iloc[0]



print('\nConclusion:')

if not ppg_row['excel_file_available'] and nvda_row['excel_file_available']:

    print('- PPG fails because its most recent 10-K currently does not expose Financial_Report.xlsx (HTTP status shown above).')

    print('- NVDA works because Financial_Report.xlsx is available, so current logic can load statement sheets.')

    print('- This is a filing-availability issue for PPG, not a confirmed keyword-matching failure in this run.')

else:

    print('- Excel availability did not differ in this run; inspect keyword/index mapping next.')

,ticker,accession,excel_url_status,excel_file_available,excel_sheet_count,statements_returned_by_current_method
0,NVDA,0001045810-25-000023,200,True,91,3
1,PPG,0000079879-26-000046,404,False,0,0



Conclusion:
- PPG fails because its most recent 10-K currently does not expose Financial_Report.xlsx (HTTP status shown above).
- NVDA works because Financial_Report.xlsx is available, so current logic can load statement sheets.
- This is a filing-availability issue for PPG, not a confirmed keyword-matching failure in this run.


## SEC directory inspection for PPG filing



This section checks the SEC directory contents directly and validates whether `Financial_Report.xlsx` exists for the latest PPG 10-K accession.

In [9]:
import requests

from bs4 import BeautifulSoup



ppg_accession = results['PPG']['diag']['accession'].iloc[0]

ppg_client = EdgarFinancials(email=EMAIL, ticker='PPG')

ppg_cik = ppg_client.company.cik

acc_no_hyphens = ppg_accession.replace('-', '')

base_url = f'https://www.sec.gov/Archives/edgar/data/{ppg_cik}/{acc_no_hyphens}'



index_json_url = f'{base_url}/index.json'

index_html_url = f'{base_url}/'

xlsx_url = f'{base_url}/Financial_Report.xlsx'

filing_summary_url = f'{base_url}/FilingSummary.xml'



index_json = requests.get(index_json_url, headers=ppg_client.header, timeout=30).json()

dir_items = index_json.get('directory', {}).get('item', [])

names = [item.get('name', '') for item in dir_items]



print(f'PPG accession: {ppg_accession}')

print(f'Directory URL: {index_html_url}')

print(f'Financial_Report.xlsx present in index.json? {"Financial_Report.xlsx" in names}')

print(f'Financial_Report.xlsx HTTP status: {requests.get(xlsx_url, headers=ppg_client.header, timeout=30).status_code}')

print('Core files present:')

for fname in ['FilingSummary.xml', 'MetaLinks.json', 'ppg-20251231.htm', 'ppg-20251231_pre.xml', '0000079879-26-000046-xbrl.zip']:

    print(f'  - {fname}: {fname in names}')



xml_text = requests.get(filing_summary_url, headers=ppg_client.header, timeout=30).text

soup = BeautifulSoup(xml_text, 'xml')

statement_reports = []

for report in soup.find_all('Report'):

    menu = (report.find('MenuCategory').text if report.find('MenuCategory') else '').strip()

    short = (report.find('ShortName').text if report.find('ShortName') else '').strip()

    html_file = (report.find('HtmlFileName').text if report.find('HtmlFileName') else '').strip()

    if menu == 'Statements':

        statement_reports.append((short, html_file))



print('\nStatements listed in FilingSummary.xml:')

for short, html_file in statement_reports:

    print(f'  - {short} -> {html_file}')

PPG accession: 0000079879-26-000046
Directory URL: https://www.sec.gov/Archives/edgar/data/79879/000007987926000046/
Financial_Report.xlsx present in index.json? False
Financial_Report.xlsx HTTP status: 404
Core files present:
  - FilingSummary.xml: True
  - MetaLinks.json: True
  - ppg-20251231.htm: True
  - ppg-20251231_pre.xml: True
  - 0000079879-26-000046-xbrl.zip: True

Statements listed in FilingSummary.xml:
  - Consolidated Statement of Income -> R3.htm
  - Consolidated Statement of Comprehensive Income -> R4.htm
  - Consolidated Balance Sheet -> R5.htm
  - Consolidated Statement of Shareholders' Equity -> R6.htm
  - Consolidated Statement of Cash Flows -> R7.htm


In [11]:
# Fallback test: pull statement tables from FilingSummary-linked HTML pages (R*.htm)

import pandas as pd

import io



fallback_rows = []

statement_targets = {

    'Income Statement': ['income'],

    'Balance Sheet': ['balance'],

    'Cash Flow Statement': ['cash flow']

}



for statement_type, keys in statement_targets.items():

    matched = None

    for short, html_file in statement_reports:

        short_l = short.lower()

        if any(k in short_l for k in keys):

            matched = (short, html_file)

            break



    if not matched:

        fallback_rows.append({

            'statement_type': statement_type,

            'short_name': None,

            'html_file': None,

            'http_status': None,

            'tables_detected': 0,

            'fallback_status': 'no_statement_match_in_filing_summary'

        })

        continue



    short_name, html_file = matched

    statement_url = f'{base_url}/{html_file}'



    try:

        resp = requests.get(statement_url, headers=ppg_client.header, timeout=30)

        status = resp.status_code

        resp.raise_for_status()



        # Parse from in-memory HTML so SEC request headers are applied

        html_tables = pd.read_html(io.StringIO(resp.text))



        fallback_rows.append({

            'statement_type': statement_type,

            'short_name': short_name,

            'html_file': html_file,

            'http_status': status,

            'tables_detected': len(html_tables),

            'fallback_status': 'ok' if len(html_tables) > 0 else 'no_tables_parsed'

        })

    except Exception as ex:

        fallback_rows.append({

            'statement_type': statement_type,

            'short_name': short_name,

            'html_file': html_file,

            'http_status': status if 'status' in locals() else None,

            'tables_detected': 0,

            'fallback_status': f'error: {type(ex).__name__}'

        })



fallback_df = pd.DataFrame(fallback_rows)

display(fallback_df)



if (fallback_df['fallback_status'] == 'ok').all():

    print('\nFallback path is viable for this filing: statements can be pulled from FilingSummary-linked HTML even without Financial_Report.xlsx.')

else:

    print('\nFallback path is partial/failed for at least one statement; inspect fallback_df for details.')

,statement_type,short_name,html_file,http_status,tables_detected,fallback_status
0,Income Statement,Consolidated Statement of Income,R3.htm,200,31,ok
1,Balance Sheet,Consolidated Balance Sheet,R5.htm,200,41,ok
2,Cash Flow Statement,Consolidated Statement of Cash Flows,R7.htm,200,51,ok



Fallback path is viable for this filing: statements can be pulled from FilingSummary-linked HTML even without Financial_Report.xlsx.


## Convert statement HTML into Excel-like DataFrame format



This section converts statement pages (`R*.htm`) into a normalized DataFrame shape intended to match the cleaned Excel statement DataFrames returned by the current pipeline.

In [15]:
import re

import io

import requests

from bs4 import BeautifulSoup

import pandas as pd



def _flatten_columns(columns):

    if isinstance(columns, pd.MultiIndex):

        out = []

        for tup in columns:

            parts = [str(x).strip() for x in tup if str(x).strip() and str(x).strip().lower() != 'nan']

            out.append(' '.join(parts).strip())

        return out

    return [str(c).strip() for c in columns]



def _dedupe_repeated_header(text):

    txt = re.sub(r'\s+', ' ', str(text)).strip()

    tokens = txt.split(' ')

    if len(tokens) >= 4 and len(tokens) % 2 == 0:

        mid = len(tokens) // 2

        if tokens[:mid] == tokens[mid:]:

            return ' '.join(tokens[:mid])

    return txt



def _coerce_numeric_like(series):

    def _parse(v):

        if pd.isna(v):

            return v

        s = str(v).strip()

        if s == '' or s.lower() in ['none', 'nan']:

            return pd.NA

        neg = s.startswith('(') and s.endswith(')')

        s = s.replace('(', '').replace(')', '')

        s = s.replace('$', '').replace(',', '').replace('%', '').strip()

        if re.fullmatch(r'-?\d+(\.\d+)?', s):

            num = float(s) if '.' in s else int(s)

            return -num if neg else num

        return v

    return series.map(_parse)



def _normalize_statement_like_excel(df_raw):

    df = df_raw.copy()

    df.columns = _flatten_columns(df.columns)

    df.columns = [_dedupe_repeated_header(c) for c in df.columns]

    df.columns = [re.sub(r'\s+', ' ', c).strip() for c in df.columns]



    unnamed_cols = [i for i, c in enumerate(df.columns) if c.lower().startswith('unnamed')]

    if len(unnamed_cols) > 0 and len(df) > 0:

        first_row = df.iloc[0].fillna('')

        new_cols = []

        for i, c in enumerate(df.columns):

            if i in unnamed_cols:

                new_cols.append(str(first_row.iloc[i]).strip())

            else:

                suffix = str(first_row.iloc[i]).strip()

                new_cols.append(f'{c} {suffix}'.strip() if suffix else c)

        df.columns = [_dedupe_repeated_header(re.sub(r'\s+', ' ', c).strip()) for c in new_cols]

        df = df.iloc[1:].reset_index(drop=True)



    # Remove fully-empty columns and rows

    df = df.dropna(axis=1, how='all')

    df = df.dropna(axis=0, how='all').reset_index(drop=True)



    # Apply same header clean-up utility used by Excel path

    df = EdgarFinancials.clean_dataframe_header(df)

    df.columns = [_dedupe_repeated_header(str(c)) for c in df.columns]



    # Coerce numeric-like values in value/date columns to align with Excel output style

    if df.shape[1] > 1:

        for col in df.columns[1:]:

            df[col] = _coerce_numeric_like(df[col])



    return df



def _statement_file_from_filing_summary(client, filing, statement_type):

    statement_keywords_local = {

        'Income Statement': ['income'],

        'Balance Sheet': ['balance'],

        'Cash Flow Statement': ['cash flow', 'cash flows']

    }

    cik = client.company.cik

    accession_no_hyphens = filing.accession_number.replace('-', '')

    base_url_local = f'https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_hyphens}'

    summary_url = f'{base_url_local}/FilingSummary.xml'



    xml_text_local = requests.get(summary_url, headers=client.header, timeout=30).text

    soup_local = BeautifulSoup(xml_text_local, 'xml')



    keys = statement_keywords_local[statement_type]

    for report in soup_local.find_all('Report'):

        menu = (report.find('MenuCategory').text if report.find('MenuCategory') else '').strip()

        short = (report.find('ShortName').text if report.find('ShortName') else '').strip()

        html_file = (report.find('HtmlFileName').text if report.find('HtmlFileName') else '').strip()

        if menu == 'Statements' and any(k in short.lower() for k in keys):

            return base_url_local, short, html_file

    return base_url_local, None, None



def html_statement_to_excel_like_df(client, filing, statement_type):

    base_url_local, short_name, html_file = _statement_file_from_filing_summary(client, filing, statement_type)

    if not html_file:

        raise ValueError(f'No statement HTML file found for {statement_type}')



    statement_url = f'{base_url_local}/{html_file}'

    resp = requests.get(statement_url, headers=client.header, timeout=30)

    resp.raise_for_status()

    tables = pd.read_html(io.StringIO(resp.text))



    # Choose the widest non-empty table as the primary statement grid

    candidate_tables = [t for t in tables if t.shape[0] > 3 and t.shape[1] > 1]

    if not candidate_tables:

        raise ValueError(f'No parseable statement table for {statement_type}')

    table = sorted(candidate_tables, key=lambda x: (x.shape[1], x.shape[0]), reverse=True)[0]



    normalized = _normalize_statement_like_excel(table)

    return short_name, html_file, normalized



def excel_statement_df(client, filing, statement_type):

    idx_map = client.find_statement_indices_by_keywords(filing, [statement_type])

    if statement_type not in idx_map:

        raise ValueError(f'Statement index not found for {statement_type}')

    sheets = client.get_excel_from_filing(filing)

    idx = idx_map[statement_type]

    if idx >= len(sheets):

        raise ValueError(f'Index {idx} out of range for Excel sheet list')

    sheet_name, df = sheets[idx]

    cleaned = client.clean_dataframe_header(df)

    return sheet_name, cleaned



print('HTML->Excel-like conversion helpers ready.')

HTML->Excel-like conversion helpers ready.


In [16]:
# Test 1: NVDA benchmark -> compare Excel vs HTML-converted format for Income Statement

nvda_client = EdgarFinancials(email=EMAIL, ticker='NVDA')

nvda_filing = nvda_client.get_multiple_filings('10-K', 1)[0]



nvda_excel_sheet, nvda_excel_df = excel_statement_df(nvda_client, nvda_filing, 'Income Statement')

nvda_html_short, nvda_html_file, nvda_html_df = html_statement_to_excel_like_df(nvda_client, nvda_filing, 'Income Statement')



comparison = pd.DataFrame([

    {

        'source': 'NVDA Excel',

        'name': nvda_excel_sheet,

        'rows': nvda_excel_df.shape[0],

        'cols': nvda_excel_df.shape[1]

    },

    {

        'source': 'NVDA HTML converted',

        'name': f'{nvda_html_short} ({nvda_html_file})',

        'rows': nvda_html_df.shape[0],

        'cols': nvda_html_df.shape[1]

    }

])

display(comparison)



print('NVDA Excel columns (first 6):', list(nvda_excel_df.columns[:6]))

print('NVDA HTML-converted columns (first 6):', list(nvda_html_df.columns[:6]))



display(nvda_excel_df.head(8))

display(nvda_html_df.head(8))

,source,name,rows,cols
0,NVDA Excel,Consolidated Statements of Inco,23,4
1,NVDA HTML converted,Consolidated Statements of Income (R3.htm),23,4


NVDA Excel columns (first 6): ['Consolidated Statements of Income - USD ($) shares in Millions, $ in Millions', '12 Months Ended Jan. 26, 2025', 'Jan. 28, 2024', 'Jan. 29, 2023']
NVDA HTML-converted columns (first 6): ['Consolidated Statements of Income - USD ($) shares in Millions, $ in Millions', '12 Months Ended Jan. 26, 2025', '12 Months Ended Jan. 28, 2024', '12 Months Ended Jan. 29, 2023']


,"Consolidated Statements of Income - USD ($) shares in Millions, $ in Millions","12 Months Ended Jan. 26, 2025","Jan. 28, 2024","Jan. 29, 2023"
0,Income Statement [Abstract],,,
1,Revenue,130497,60922,26974
2,Cost of revenue,32639,16621,11618
3,Gross profit,97858,44301,15356
4,Operating expenses,,,
5,Research and development,12914,8675,7339
6,"Sales, general and administrative",3491,2654,2440
7,Acquisition termination cost,0,0,1353


,"Consolidated Statements of Income - USD ($) shares in Millions, $ in Millions","12 Months Ended Jan. 26, 2025","12 Months Ended Jan. 28, 2024","12 Months Ended Jan. 29, 2023"
0,Income Statement [Abstract],NaN,NaN,NaN
1,Revenue,130497.0,60922.0,26974.0
2,Cost of revenue,32639.0,16621.0,11618.0
3,Gross profit,97858.0,44301.0,15356.0
4,Operating expenses,NaN,NaN,NaN
5,Research and development,12914.0,8675.0,7339.0
6,"Sales, general and administrative",3491.0,2654.0,2440.0
7,Acquisition termination cost,0.0,0.0,1353.0


In [17]:
# Test 2: PPG fallback -> produce Excel-like DataFrames from HTML statements

ppg_client = EdgarFinancials(email=EMAIL, ticker='PPG')

ppg_filing = ppg_client.get_multiple_filings('10-K', 1)[0]



ppg_out = []

ppg_statement_dfs = {}

for stype in ['Income Statement', 'Balance Sheet', 'Cash Flow Statement']:

    short_name, html_file, ppg_df = html_statement_to_excel_like_df(ppg_client, ppg_filing, stype)

    ppg_statement_dfs[stype] = ppg_df

    ppg_out.append({

        'statement_type': stype,

        'source_short_name': short_name,

        'html_file': html_file,

        'rows': ppg_df.shape[0],

        'cols': ppg_df.shape[1],

        'first_columns': ' | '.join([str(c) for c in ppg_df.columns[:4]])

    })



display(pd.DataFrame(ppg_out))

for stype, ppg_df in ppg_statement_dfs.items():

    print('\n' + '='*90)

    print(stype)

    display(ppg_df.head(8))

,statement_type,source_short_name,html_file,rows,cols,first_columns
0,Income Statement,Consolidated Statement of Income,R3.htm,30,4,Consolidated Statement of Income - USD ($) $ i...
1,Balance Sheet,Consolidated Balance Sheet,R5.htm,40,3,Consolidated Balance Sheet - USD ($) $ in Mill...
2,Cash Flow Statement,Consolidated Statement of Cash Flows,R7.htm,51,4,Consolidated Statement of Cash Flows - USD ($)...



Income Statement


,Consolidated Statement of Income - USD ($) $ in Millions,"12 Months Ended Dec. 31, 2025","12 Months Ended Dec. 31, 2024","12 Months Ended Dec. 31, 2023"
0,Income Statement [Abstract],NaN,NaN,NaN
1,Net sales,15875.0,15845.0,16242.0
2,"Cost of sales, exclusive of depreciation and a...",9316.0,9252.0,9678.0
3,"Selling, general and administrative",3439.0,3391.0,3401.0
4,Depreciation,403.0,360.0,360.0
5,Amortization,125.0,132.0,154.0
6,"Research and development, net",423.0,423.0,424.0
7,Interest expense,241.0,241.0,247.0



Balance Sheet


,Consolidated Balance Sheet - USD ($) $ in Millions,"Dec. 31, 2025","Dec. 31, 2024"
0,Current assets,NaN,NaN
1,Cash and cash equivalents,2163.0,1270.0
2,Short-term investments,56.0,88.0
3,Receivables,3336.0,2985.0
4,Inventories,1996.0,1846.0
5,Other current assets,408.0,368.0
6,Total current assets,7959.0,6557.0
7,"Property, plant and equipment, net",4005.0,3464.0



Cash Flow Statement


,Consolidated Statement of Cash Flows - USD ($) $ in Millions,"12 Months Ended Dec. 31, 2025","12 Months Ended Dec. 31, 2024","12 Months Ended Dec. 31, 2023"
0,Operating activities,NaN,NaN,NaN
1,Income from continuing operations,1587.0,1377.0,1262.0
2,Adjustments to reconcile net income to cash fr...,NaN,NaN,NaN
3,Depreciation and amortization,528.0,492.0,514.0
4,Pension settlement charge,0.0,0.0,190.0
5,"Impairment and other-related charges, net",24.0,146.0,160.0
6,Stock-based compensation expense,46.0,42.0,56.0
7,Deferred income taxes,-27.0,-97.0,-187.0
